# 02 — Supervisado: qué representación, qué reducción y con qué umbral

## 0. Entorno

Una sola variable de ruta. Los escenarios se cargan por separado y nunca se mezclan.


In [7]:
%matplotlib inline
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src2.data import eliminar_columnas
from src2.features import rango_escenarios, rango_movil
from src2.split import separar_variables, aplanar_escenarios, aplanar_ventanas
from src2.train import especificidad_s0, resumen_representaciones, rf_CV, rf_LOSO

pd.set_option("display.width", 180)

BASE = Path("/media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/datasets/")

ARCHIVOS = {
    "S1": "datasetS1.csv", "S1alt": "datasetS1alt.csv", "S2": "datasetS2.csv",
    "S3": "datasetS3.csv", "S4": "datasetS4.csv", "S5": "datasetS5.csv",
    "S6": "datasetS6.csv",
}


In [8]:
brutos = {k: pd.read_csv(BASE / f) for k, f in ARCHIVOS.items()}
s0_bruto = pd.read_csv(BASE / "datasetS0.csv")

dfs = {k: eliminar_columnas(df) for k, df in brutos.items()}
s0 = eliminar_columnas(s0_bruto)

X, Y = {}, {}
for k, df in dfs.items():
    X[k], Y[k] = separar_variables(df)

X_s0, _ = separar_variables(s0)
print("escenarios:", list(dfs), "| features:", X["S1"].shape[1])


escenarios: ['S1', 'S1alt', 'S2', 'S3', 'S4', 'S5', 'S6'] | features: 132


## 1. La fuga de `Profile.*`

`Profile.*` no es una medición: es el guion del escenario. Si S0 se grabó con el perfil congelado y
los escenarios de ataque con el perfil variable, la columna identifica el escenario. Por eso
`eliminar_columnas` la descarta.


In [9]:
perfil = {}
for nombre, df in {**brutos, "S0": s0_bruto}.items():
    perfil[nombre] = {
        "desvio": df["Profile.pv_active_kw"].std(),
        "minimo": df["Profile.pv_active_kw"].min(),
        "maximo": df["Profile.pv_active_kw"].max(),
    }
display(pd.DataFrame(perfil).T.round(4))


,desvio,minimo,maximo
S1,5.6355,-28.9036,-4.5906
S1alt,5.6940,-28.9036,-4.5906
S2,5.6617,-28.9036,-4.5906
S3,5.7072,-28.9036,-4.6188
S4,5.9619,-28.9036,-4.5906
S5,6.4944,-28.7172,-5.5459
S6,5.7457,-28.9036,-4.5906
S0,0.0000,-20.4093,-20.4093


## 2. Protocolo

**Por qué LOSO.** Un split aleatorio en datos de 1 Hz es una trampa: dos filas consecutivas son casi
la misma fila, así que el test queda lleno de copias del entrenamiento. La comparación de abajo lo
muestra con el mismo modelo y las mismas filas.

**Con qué se mide.** AUC por fold y `recall@5` (fracción de ataques en el 5 % de mayor score, la
convención de `src/train.py`). La especificidad se informa a TPR fijo, no a 0.5: un modelo puede
ordenar bien y fallar el corte.


In [14]:
tabla_cv = rf_CV(X, Y, 8)
tabla_loso = rf_LOSO(X, Y)

print("AUC medio, CV aleatoria:", round(tabla_cv.loc["mean", "auc"], 3))
print("AUC medio, LOSO:", round(tabla_loso.auc.mean(), 3))
display(tabla_loso)
display(tabla_cv)

RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced
AUC medio, CV aleatoria: 1.0
AUC medio, LOSO: 0.865


,n,positivos,accuracy,precision,recall,f1,auc,recall_5,features
escenario,,,,,,,,,
S1,1798,913,0.519,1.000,0.054,0.102,0.920,0.499,132
S1alt,1792,910,0.748,0.669,1.000,0.801,0.820,0.230,132
S2,1768,903,0.589,0.656,0.412,0.506,0.725,0.016,132
S3,1800,912,0.794,0.759,0.872,0.811,0.850,0.182,132
S4,1798,1257,0.632,0.927,0.514,0.661,0.901,0.346,132
S5,1800,1341,0.647,0.943,0.559,0.702,0.902,0.242,132
S6,1800,1350,0.571,0.972,0.441,0.607,0.936,0.658,132


,accuracy,precision,recall,f1,auc
fold,,,,,
1,0.994,0.993,0.998,0.995,1.000
2,0.994,0.998,0.993,0.995,0.999
3,0.994,0.998,0.993,0.995,1.000
4,0.999,0.999,0.999,0.999,1.000
5,0.996,0.998,0.995,0.996,1.000
6,0.999,1.000,0.999,0.999,1.000
7,0.997,0.999,0.997,0.998,1.000
8,0.996,0.999,0.994,0.996,1.000
mean,0.996,0.998,0.996,0.997,1.000


## 3. Las tres representaciones

- **crudo**: una fila es un segundo, 132 features.
- **rango**: una fila es un segundo, 132 features de amplitud.
- **ventana**: **una muestra son 20 segundos**, 132 × 20 = 2640 features aplanadas. Pierde las
  primeras 19 filas de cada escenario y hereda la etiqueta de su fila de cierre.


In [11]:
X_rango = rango_escenarios(X)
X_ventana, Y_ventana = aplanar_escenarios(X, Y, 20)

X_s0_rango = rango_movil(X_s0)
y_s0 = pd.Series(0, index=X_s0.index)
X_s0_ventana, _ = aplanar_ventanas(X_s0, y_s0, 20)


In [12]:
tabla_crudo = rf_LOSO(X, Y)
tabla_rango = rf_LOSO(X_rango, Y)
tabla_ventana = rf_LOSO(X_ventana, Y_ventana)

espec_crudo = especificidad_s0(X, Y, X_s0)
espec_rango = especificidad_s0(X_rango, Y, X_s0_rango)
espec_ventana = especificidad_s0(X_ventana, Y_ventana, X_s0_ventana)

resumen = resumen_representaciones(
    {"crudo": tabla_crudo, "rango": tabla_rango, "ventana": tabla_ventana},
    {"crudo": espec_crudo, "rango": espec_rango, "ventana": espec_ventana},
)
display(resumen)


RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced
RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced
RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced


,features,muestras,AUC medio,desvío,recall@5,espec S0 @TPR95
representación,,,,,,
crudo,132,12556,0.865,0.074,0.310,0.991
rango,132,12556,0.876,0.058,0.490,0.954
ventana,2640,12423,0.836,0.131,0.339,0.874


### 3.1 El reloj del dispositivo

`Substation-AC-Meter.realtime` es el reloj interno del equipo. Aplanado en 20 pasos entrega 20 valores
que codifican la posición de la muestra en la grabación. La prueba es una sola columna: la misma
ventana con el contador y sin él.


In [13]:
X_reloj = {k: X[k].assign(realtime=brutos[k]["Substation-AC-Meter.realtime"]) for k in X}
s0_reloj = X_s0.assign(realtime=s0_bruto["Substation-AC-Meter.realtime"])

X_ventana_reloj, Y_ventana_reloj = aplanar_escenarios(X_reloj, Y, 20)
X_s0_ventana_reloj, _ = aplanar_ventanas(s0_reloj, y_s0, 20)

tabla_reloj = rf_LOSO(X_ventana_reloj, Y_ventana_reloj)
espec_reloj = especificidad_s0(X_ventana_reloj, Y_ventana_reloj, X_s0_ventana_reloj)

print("ventana sin reloj:", round(tabla_ventana.auc.mean(), 3), "| espec S0", round(espec_ventana, 3))
print("ventana con reloj:", round(tabla_reloj.auc.mean(), 3), "| espec S0", round(espec_reloj, 3))


/tmp/ipykernel_3570316/972974418.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_reloj = {k: X[k].assign(realtime=brutos[k]["Substation-AC-Meter.realtime"]) for k in X}
/tmp/ipykernel_3570316/972974418.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  s0_reloj = X_s0.assign(realtime=s0_bruto["Substation-AC-Meter.realtime"])


RF LOSO: 120 arboles, depth 18, leaf 2, class_weight=balanced
ventana sin reloj: 0.836 | espec S0 0.874
ventana con reloj: 0.792 | espec S0 1.0


## Lo que falta

- **4. Reducción** sobre la representación elegida: `ninguna`, `corr`, `pca`, `block_pca`, ajustados
  por fold, con la tabla de interacción sobre las otras dos representaciones.
- **5. Modelo**: Random Forest contra XGBoost sobre la configuración elegida.
- **6. Punto de operación**: umbral por fold a TPR objetivo, especificidad S0 a 90/95/99 %.
- **7. Auditoría**: validación anidada, porque elegir el máximo entre varias configuraciones sobre
  7 folds es optimista.
- **8. Producción**: reentrenar con todo, guardar y predecir un escenario nuevo.
